# Regresion Supervisada MP2.5 a 24 horas

Este notebook deja la **logica de negocio en `src/regression.py`** y usa las celdas solo como demostracion del flujo:

Carga -> EDA rapida -> variable objetivo -> ingenieria de caracteristicas -> entrenamiento -> comparacion -> real vs predicho -> guardado de artefactos.

In [ ]:
import os
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
from IPython.display import display


def es_raiz_proyecto(path: Path) -> bool:
    return (
        (path / "docker-compose.yml").is_file()
        and (path / "src").is_dir()
        and (path / "notebooks").is_dir()
    )


def encontrar_raiz_proyecto() -> Path:
    candidatos = []

    for base in [Path.cwd(), *Path.cwd().parents]:
        candidatos.append(base)

    for env_key in ("__vsc_ipynb_file__", "JPY_SESSION_NAME", "PWD", "VSCODE_CWD"):
        valor = os.environ.get(env_key)
        if not valor:
            continue

        ruta = Path(valor).expanduser()
        if ruta.suffix == ".ipynb":
            ruta = ruta.parent

        candidatos.extend([ruta, *ruta.parents])

    revisados = set()
    for candidato in candidatos:
        candidato = candidato.resolve()
        if candidato in revisados:
            continue
        revisados.add(candidato)
        if es_raiz_proyecto(candidato):
            return candidato

    raise FileNotFoundError(
        "No se pudo ubicar la raiz del proyecto desde el notebook. "
        "Abre Jupyter desde la carpeta del repo o ejecuta el notebook dentro del workspace."
    )


PROJECT_ROOT = encontrar_raiz_proyecto()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.regression import (
    DEFAULT_DATASET_PATH,
    DEFAULT_METRICS_PATH,
    DEFAULT_MODEL_PATH,
    DEFAULT_PREDICTIONS_PATH,
    TARGET_HORIZON_HOURS,
    TARGET_SHIFT_STEPS,
    SAMPLING_FREQUENCY_HOURS,
    cargar_dataset,
    crear_variable_objetivo,
    ingenieria_caracteristicas,
    filtrar_dataset_entrenamiento,
    split_temporal,
    entrenar_modelos,
    seleccionar_mejor_modelo,
    evaluar_modelo,
    guardar_metricas,
    guardar_modelo,
    guardar_predicciones,
    construir_predicciones_evaluacion,
    construir_pronosticos_24h,
    obtener_importancia_variables,
)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATASET_PATH: {DEFAULT_DATASET_PATH}")

## 1. Carga del dataset

In [ ]:
dataset = cargar_dataset(DEFAULT_DATASET_PATH)

print("Filas:", len(dataset))
print("Columnas:", len(dataset.columns))
display(dataset.head())

## 2. EDA rapida

In [ ]:
resumen_eda = dataset[[
    "mp25",
    "mp10",
    "so2",
    "no2",
    "temperatura",
    "humedad",
    "velocidad_viento",
]].describe().T

conteo_contexto = pd.DataFrame(
    {
        "estaciones": [dataset["codigo_estacion"].nunique()],
        "comunas": [dataset["comuna"].nunique()],
        "regiones": [dataset["region"].nunique()],
    }
)

display(conteo_contexto)
display(resumen_eda)

## 3. Variable objetivo e ingenieria de caracteristicas

In [ ]:
dataset_target = crear_variable_objetivo(dataset, target_shift_steps=TARGET_SHIFT_STEPS)
dataset_features = ingenieria_caracteristicas(dataset_target)

columnas_clave = [
    "fecha_hora",
    "fecha_hora_objetivo",
    "codigo_estacion",
    "mp25",
    "mp25_24h_estacion",
    "mp25_24h_comuna",
    "mp25_24h",
    "lag_1",
    "lag_2",
    "lag_3",
    "delta_mp25",
    "rolling_mean_6",
    "rolling_mean_12",
    "estacion_del_ano",
]

display(dataset_features[columnas_clave].head(10))

## 4. Split temporal de entrenamiento y prueba

In [ ]:
dataset_entrenamiento = filtrar_dataset_entrenamiento(dataset_features)
train_df, test_df = split_temporal(dataset_entrenamiento)

resumen_split = pd.DataFrame(
    [
        {
            "dataset": "train",
            "filas": len(train_df),
            "fecha_min": train_df["fecha_hora"].min(),
            "fecha_max": train_df["fecha_hora"].max(),
        },
        {
            "dataset": "test",
            "filas": len(test_df),
            "fecha_min": test_df["fecha_hora"].min(),
            "fecha_max": test_df["fecha_hora"].max(),
        },
    ]
)

display(resumen_split)

## 5. Entrenamiento y comparacion de modelos

In [ ]:
resultados = entrenar_modelos(train_df, test_df)
metricas_df = pd.DataFrame(
    {nombre: resultado.metricas for nombre, resultado in resultados.items()}
).T.sort_values("rmse")

metricas_df.index.name = "modelo"
display(metricas_df)

## 6. Seleccion del mejor modelo

In [ ]:
mejor_modelo = seleccionar_mejor_modelo({nombre: resultado.metricas for nombre, resultado in resultados.items()})
mejor_pipeline = resultados[mejor_modelo].pipeline

print("Mejor modelo:", mejor_modelo)
print("Metricas holdout:")
display(pd.DataFrame([resultados[mejor_modelo].metricas]))

## 7. Real vs Predicho

In [ ]:
evaluacion = construir_predicciones_evaluacion(
    test_df,
    resultados[mejor_modelo].predicciones_test,
)

display(evaluacion.head())

fig = px.scatter(
    evaluacion,
    x="mp25_real_24h",
    y="mp25_predicho_24h",
    color="comuna",
    hover_data=["codigo_estacion", "fecha_hora_base", "fecha_hora_objetivo"],
    title="Comparacion Real vs Predicho (holdout)",
)

minimo = min(evaluacion["mp25_real_24h"].min(), evaluacion["mp25_predicho_24h"].min())
maximo = max(evaluacion["mp25_real_24h"].max(), evaluacion["mp25_predicho_24h"].max())
fig.add_shape(
    type="line",
    x0=minimo,
    y0=minimo,
    x1=maximo,
    y1=maximo,
    line={"dash": "dash"},
)
fig.show()

## 8. Importancia de variables

In [ ]:
importancia_df = obtener_importancia_variables(mejor_pipeline, top_n=15)
display(importancia_df)

## 9. Guardar modelo, metricas y predicciones

In [ ]:
pronosticos_24h = construir_pronosticos_24h(dataset_features, mejor_pipeline)
predicciones_finales = pd.concat([evaluacion, pronosticos_24h], ignore_index=True)

metricas_payload = {
    "horizon_hours": TARGET_HORIZON_HOURS,
    "sampling_frequency_hours": SAMPLING_FREQUENCY_HOURS,
    "target_shift_steps": TARGET_SHIFT_STEPS,
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "best_model": mejor_modelo,
    "models": {
        nombre: resultado.metricas for nombre, resultado in resultados.items()
    },
}

ruta_modelo = guardar_modelo(mejor_pipeline, DEFAULT_MODEL_PATH)
ruta_metricas = guardar_metricas(metricas_payload, DEFAULT_METRICS_PATH)
ruta_predicciones = guardar_predicciones(predicciones_finales, DEFAULT_PREDICTIONS_PATH)

resumen_salida = pd.DataFrame(
    [
        {"artefacto": "modelo", "ruta": str(ruta_modelo)},
        {"artefacto": "metricas", "ruta": str(ruta_metricas)},
        {"artefacto": "predicciones", "ruta": str(ruta_predicciones)},
    ]
)

display(resumen_salida)
display(predicciones_finales.head())